In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [2]:
from src.data_utils import preprocess_dataset, split_dataset

# Пути
raw_path = "data/raw_dataset.csv"
processed_path = "data/dataset_processed.csv"
data_dir = "data"

# Очистка
if not os.path.exists(processed_path):
    preprocess_dataset(raw_path, processed_path)

# Разбиение
if not os.path.exists("data/train.csv"):
    split_dataset(processed_path, data_dir)

In [3]:
tokenizer_lstm = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [4]:
from torch.utils.data import DataLoader
from src.next_token_dataset import NextTokenDataset, collate_fn

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

train_df = train_df.dropna()[:10000]
val_df = val_df.dropna()
test_df = test_df.dropna()

train_ids = tokenizer_lstm(train_df["text"].tolist(), add_special_tokens=False)["input_ids"]
val_ids = tokenizer_lstm(val_df["text"].tolist(), add_special_tokens=False)["input_ids"]
test_ids = tokenizer_lstm(test_df["text"].tolist(), add_special_tokens=False)["input_ids"]

train_dataset = NextTokenDataset(train_ids)
val_dataset = NextTokenDataset(val_ids)
test_dataset = NextTokenDataset(test_ids)

batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

In [5]:
import torch.nn as nn
from src.lstm_model import LSTMModel

lstm_model = LSTMModel(
    vocab_size=tokenizer_lstm.vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=1,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer_lstm.pad_token_id)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.002)

In [6]:
from src.lstm_train import train
n_epoch = 2
train(lstm_model, n_epoch, criterion, optimizer, tokenizer_lstm, train_loader, val_loader, device)
torch.save(lstm_model.state_dict(), "models/lstm_model.pth")

Epoch 1 | Train Loss: 6.6624 | Val Loss: 6.1904 | ROUGE-1: 0.0757 | ROUGE-2: 0.0233

=== Examples ===

Prompt:
fingernail is ok it only hurts when u press it down congrats
Target:
with explore pics told
Generated:
i ' m so much
--------------------------------------------------

Prompt:
come to mexico and i do to you all the mexican or italian food that you want ja
Target:
##ja xoxo from monterrey
Generated:
to be a good day to
--------------------------------------------------
Epoch 2 | Train Loss: 5.6202 | Val Loss: 5.9862 | ROUGE-1: 0.0600 | ROUGE-2: 0.0050

=== Examples ===

Prompt:
fingernail is ok it only hurts when u press it down congrats
Target:
with explore pics told
Generated:
to be back to work
--------------------------------------------------

Prompt:
come to mexico and i do to you all the mexican or italian food that you want ja
Target:
##ja xoxo from monterrey
Generated:
to go to work tomorrow and
--------------------------------------------------


In [7]:
import pandas as pd
from src.eval_transformer_pipeline import (
    build_transformer,
    evaluate_transformer,
    show_example_models
)
# создаем transformer
generator, tokenizer = build_transformer()

# считаем ROUGE
rouge1, rouge2 = evaluate_transformer(generator, tokenizer, val_df)

print(f"Transformer ROUGE-1: {rouge1:.4f} | ROUGE-2: {rouge2:.4f}")

# показываем примеры
show_example_models(
    generator,
    tokenizer,
    test_df,
    lstm_model,
    tokenizer_lstm,
    num_examples=3
)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are no

Transformer ROUGE-1: 0.0900 | ROUGE-2: 0.0093

===== Примеры на тестовой выборке =====



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Промпт:
good morning and thanks i might need it might have to deal

Цель:
with immigration today booooo

LSTM:
i ' m going to

TRANSFORMER:
with it.

------------------------------------------------------------

Промпт:
i love this song new day

Цель:
music video

LSTM:
i '

TRANSFORMER:
.

------------------------------------------------------------

Промпт:
fuckinn bored need sleep arghh life

Цель:
sucksss needing summer

LSTM:
i ' m going to

TRANSFORMER:


------------------------------------------------------------



Выводы:
